# CineEmbed — 02 Train AE (MVP scope)

Trains the AutoEncoder family for the **intermediate report MVP**:
- Run 3: `vanilla_ae_z64` (concat-AE baseline for architecture validation)
- Run 5: `ae_z64` (multi-modal AE at z=64)
- Run 19: `ae_z64_w1` (W1 uniform-weighting ablation)

**Compute estimate**: ~45 min total on Colab T4.
**Deferred for final report**: ae_z32, ae_z128, F1/F2 modality ablations, optional W4 stretch.

In [ ]:
import os, sys, json
from pathlib import Path

IN_COLAB = 'google.colab' in sys.modules

if IN_COLAB:
    from google.colab import drive
    drive.mount('/content/drive')

    # Auth: try GitHub token from Colab Secrets (for private repos), fall back to public URL
    try:
        from google.colab import userdata
        _token = userdata.get('GITHUB_TOKEN')
        REPO_URL = f"https://{_token}@github.com/bkaankaya/CineEmbed-A-Multi-Modal-Unsupervised-Film-Recommendation-System.git"
    except Exception:
        REPO_URL = "https://github.com/bkaankaya/CineEmbed-A-Multi-Modal-Unsupervised-Film-Recommendation-System.git"
    REPO_ROOT = Path('/content/cineembed-repo')
    ARTIFACTS = Path('/content/drive/MyDrive/CineEmbed/artifacts')

    if not REPO_ROOT.exists():
        get_ipython().system(f'git clone {REPO_URL} {REPO_ROOT}')
    else:
        # Pull latest if you've pushed updates from local
        get_ipython().system(f'cd {REPO_ROOT} && git pull -q')

    get_ipython().system(f'pip install -e {REPO_ROOT} -q')
else:
    REPO_ROOT = Path('..').resolve()
    ARTIFACTS = REPO_ROOT / 'artifacts'

sys.path.insert(0, str(REPO_ROOT / 'src'))

import numpy as np
import torch

from cineembed import data, backbone, heads, losses, train, eval as cev

DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'
print(f"Repo: {REPO_ROOT}\nArtifacts: {ARTIFACTS}\nDevice: {DEVICE}")

In [ ]:
X, feature_names = data.load_feature_matrix(ARTIFACTS / 'feature_matrix.npz')
labels = data.get_labels(ARTIFACTS / 'movies_eda_final.csv')
block_slices = data.get_block_indices(feature_names)
has_bio = X[:, block_slices['director'].start + 64].clone()

print(f"X: {X.shape}; has_bio sum: {int(has_bio.sum())}")
print(f"Labels: {[(k, len(set(v))) for k, v in labels.items()]}")
print(f"Block slices: {[(b, slc.stop - slc.start) for b, slc in block_slices.items()]}")

In [ ]:
BLOCK_DIMS = {b: (slc.stop - slc.start) for b, slc in block_slices.items()}
PROJ_DIMS = backbone.DEFAULT_PROJ_DIMS

w_blocks = losses.compute_block_weights(X.numpy(), block_slices, w_min=0.1, w_max=10.0)
print(f"Block weights (W2): {w_blocks}")

train_idx, val_idx = data.train_val_split(X.shape[0], val_frac=0.1, seed=42)
train_loader = data.make_dataloader(X, has_bio, batch_size=512, shuffle=True,
                                     indices=train_idx, block_slices=block_slices, seed=42)
val_loader = data.make_dataloader(X, has_bio, batch_size=512, shuffle=False,
                                    indices=val_idx, block_slices=block_slices, seed=42)

In [ ]:
import torch.nn as nn  # for vanilla wrap

def train_ae_run(run_name, z_dim, w_blocks_to_use, *,
                 use_proj_dims=PROJ_DIMS, mask_text=False, mask_director=False,
                 n_epochs=100, vanilla=False):
    """Train one AE run; save checkpoint and metrics. Returns metrics dict.

    Modality ablation (mask_text / mask_director) uses the backbone's `block_mask`
    parameter (zeros that block's projected encoder output) AND `weighted_recon_loss`
    `exclude_blocks` (skips that block in the reconstruction loss).
    """
    torch.manual_seed(42)
    import time as _time
    _t0 = _time.time()
    masked_blocks: set = set()
    if mask_text:
        masked_blocks.add('text')
    if mask_director:
        masked_blocks.add('director')
    mask_str = ', '.join(sorted(masked_blocks)) if masked_blocks else 'none'
    print(f"\n{'='*72}")
    print(f"[{_time.strftime('%H:%M:%S')}] BEGIN run='{run_name}' z={z_dim} "
          f"vanilla={vanilla} masks=[{mask_str}]")
    print(f"{'='*72}")

    if vanilla:
        # Single FC encoder, no modality projection — vanilla concat-AE baseline.
        # Uses the same per-block decoder as multi-modal AE for fair comparison.
        bb_fc = nn.Sequential(
            nn.Linear(564, 128), nn.ReLU(), nn.Dropout(0.2), nn.Linear(128, z_dim),
        )
        class _VanillaWrap(nn.Module):
            def __init__(self, fc, z_dim_):
                super().__init__()
                self.fc = fc
                self.latent_dim = z_dim_
                self.block_order = list(BLOCK_DIMS.keys())
            def forward(self, blocks, block_mask=None):
                # block_mask intentionally ignored — vanilla baseline doesn't support
                # ablation. F1/F2 ablations only apply to multi-modal AE.
                X_cat = torch.cat([blocks[b] for b in self.block_order], dim=1)
                return self.fc(X_cat)
        bb = _VanillaWrap(bb_fc, z_dim)
    else:
        bb = backbone.MultiModalBackbone(BLOCK_DIMS, use_proj_dims, hidden_dim=128, latent_dim=z_dim)
    head = heads.AEHead(bb, BLOCK_DIMS, use_proj_dims, hidden_dim=128)

    block_mask = ({b: 0.0 for b in masked_blocks} | {b: 1.0 for b in BLOCK_DIMS if b not in masked_blocks}
                  if masked_blocks else None)

    def loss_fn(model, batch):
        if block_mask is not None and not vanilla:
            z = model.backbone(batch['blocks'], block_mask=block_mask)
            decoded = model.decoder(z)
        else:
            decoded = model(batch['blocks'])
        return losses.weighted_recon_loss(
            decoded, batch['blocks'], batch['has_bio'], w_blocks_to_use,
            exclude_blocks=masked_blocks if masked_blocks else None,
        )

    ckpt_path = ARTIFACTS / 'models' / f'{run_name}.pt'
    history = train.train_model(
        model=head, loss_fn=loss_fn,
        train_loader=train_loader, val_loader=val_loader,
        n_epochs=n_epochs, lr=1e-3, weight_decay=1e-5,
        early_stop_patience=10, early_stop_min_delta=1e-4,
        device=DEVICE, checkpoint_path=ckpt_path, seed=42,
    )

    # Embed all films with best-checkpoint weights
    train.load_checkpoint(head, ckpt_path, device=DEVICE)
    head.eval()
    print(f"[{_time.strftime('%H:%M:%S')}] extracting embeddings on full dataset ({len(X):,} films)...")
    with torch.no_grad():
        z_full = []
        full_loader = data.make_dataloader(X, has_bio, batch_size=2048, shuffle=False,
                                             block_slices=block_slices)
        for batch in full_loader:
            batch_blocks = {k: v.to(DEVICE) for k, v in batch['blocks'].items()}
            z_full.append(head.encode(batch_blocks).cpu().numpy())
    z_all = np.concatenate(z_full, axis=0)

    # KMeans on full-data latent and L4 evaluation
    print(f"[{_time.strftime('%H:%M:%S')}] KMeans(k=21) on {z_all.shape[0]:,} latent vectors...")
    c_ids = cev.cluster_assignments_kmeans(z_all, k=21, seed=42)
    metrics = cev.evaluate_run(c_ids, labels)
    metrics['run_name'] = run_name
    metrics['n_epochs'] = history['n_epochs_completed']
    metrics['final_val_loss'] = history['final_val_loss']
    metrics['z_dim'] = z_dim
    print(f"[{run_name}] z={z_dim}  epochs={history['n_epochs_completed']}  "
          f"val_loss={history['final_val_loss']:.4f}  genre_NMI={metrics['genre_nmi']:.3f}")
    _elapsed = _time.time() - _t0
    print(f"[{_time.strftime('%H:%M:%S')}] END   run='{run_name}' in {_elapsed/60:.1f} min | "
          f"genre_NMI={metrics.get('genre_nmi', 0):.3f} "
          f"decade_NMI={metrics.get('decade_nmi', 0):.3f} "
          f"lang_NMI={metrics.get('lang_nmi', 0):.3f}")
    return metrics, z_all


In [ ]:
all_metrics = []

# Run 3: vanilla concat-AE z=64
m, _ = train_ae_run('vanilla_ae_z64', z_dim=64, w_blocks_to_use=w_blocks, vanilla=True)
all_metrics.append(m)

# Run 5: multi-modal AE z=64 (z=32, z=128 deferred to final report)
m, _ = train_ae_run('ae_z64', z_dim=64, w_blocks_to_use=w_blocks)
all_metrics.append(m)

# Run 19: W1 ablation
w_uniform = {b: 1.0 for b in w_blocks}
m, _ = train_ae_run('ae_z64_w1', z_dim=64, w_blocks_to_use=w_uniform)
all_metrics.append(m)

In [ ]:
results_path = ARTIFACTS / 'eval' / 'results.json'
results_path.parent.mkdir(parents=True, exist_ok=True)
existing = json.loads(results_path.read_text()) if results_path.exists() else {}
for m in all_metrics:
    existing[m['run_name']] = m
results_path.write_text(json.dumps(existing, indent=2))
print(f"Saved {len(all_metrics)} runs to {results_path}")

## Runs deferred to "Modeling Full" (final report phase)

This MVP notebook trained 3 of the planned 7 AE-family runs. After the intermediate
report, return here and uncomment the cells below to add:

- `ae_z32` — multi-modal AE at z=32
- `ae_z128` — multi-modal AE at z=128
- `ae_z64_no_text` — F1 ablation (text modality removed)
- `ae_z64_no_director` — F2 ablation (director modality removed)
- `ae_z64_w4` — W4 Kendall learned uncertainty (optional stretch)

Code for these runs is in the implementation plan §9 (lines describing Cell 6).

In [ ]:
# ─── Deferred for final report — uncomment to run after intermediate ─────
# # ae_z32 + ae_z128
# for z in [32, 128]:
#     m, _ = train_ae_run(f'ae_z{z}', z_dim=z, w_blocks_to_use=w_blocks)
#     all_metrics.append(m)
#
# # F1 — no text
# m, _ = train_ae_run('ae_z64_no_text', z_dim=64, w_blocks_to_use=w_blocks, mask_text=True)
# all_metrics.append(m)
#
# # F2 — no director
# m, _ = train_ae_run('ae_z64_no_director', z_dim=64, w_blocks_to_use=w_blocks, mask_director=True)
# all_metrics.append(m)
#
# # Optional: W4 Kendall stretch
# # from cineembed.losses import LearnedWeightedLoss
# # bb = backbone.MultiModalBackbone(BLOCK_DIMS, PROJ_DIMS, hidden_dim=128, latent_dim=64)
# # head = heads.AEHead(bb, BLOCK_DIMS, PROJ_DIMS, hidden_dim=128)
# # learned_loss = LearnedWeightedLoss(list(BLOCK_DIMS.keys()))
# # def loss_fn(model, batch):
# #     decoded = model(batch['blocks'])
# #     return learned_loss(decoded, batch['blocks'], batch['has_bio'])
# # history = train.train_model(model=head, loss_fn=loss_fn,
# #     train_loader=train_loader, val_loader=val_loader, n_epochs=100, lr=1e-3,
# #     device=DEVICE, checkpoint_path=ARTIFACTS/'models'/'ae_z64_w4.pt',
# #     extra_params=list(learned_loss.parameters()), seed=42)